# Lunar site Sun/Earth ephemerides (DE421)

Produces observer-relative body-center vectors for Zarina's visibility engine. It does not calculate visibility windows; `analyze` owns that step. Upload Lucas's schema v2 horizon JSON files before running. No site coordinates are invented here.

Contract: `MOON_ME_DE421`; `spkcpo` / `CN+S` / `OBSERVER`; 5-minute epochs from 2026-11-14 00:00 UTC through 2026-12-14 00:00 UTC inclusive (8641); vectors in metres, unnormalized, shape `(N,3)`.

The pasted contract's `sun_vectors_may` likely contains a typo. This notebook emits `sun_vectors_m`; confirm the exact key expected by `analyze` before wiring it in.

In [ ]:
%pip -q install spiceypy numpy

In [ ]:
from pathlib import Path
import json, urllib.request
import numpy as np
import spiceypy as spice

KERNEL_DIR = Path('kernels')
KERNEL_DIR.mkdir(exist_ok=True)
KERNEL_URLS = {
    'naif0012.tls': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls',
    'de421.bsp': 'https://naif.jpl.nasa.gov/pub/naif/MAVEN/kernels/spk/de421.bsp',
    'pck00011.tpc': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc',
    'moon_pa_de421_1900-2050.bpc': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/moon_pa_de421_1900-2050.bpc',
    'moon_080317.tf': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/fk/satellites/moon_080317.tf',
}
for filename, url in KERNEL_URLS.items():
    dest = KERNEL_DIR / filename
    if not dest.exists():
        print('Downloading', filename)
        urllib.request.urlretrieve(url, dest)
    spice.furnsh(str(dest))
assert spice.namfrm('MOON_ME_DE421') != 0, 'Kernel set did not define MOON_ME_DE421'
print('Loaded:', ', '.join(KERNEL_URLS))

## Load Lucas's observer
Use the latest schema v2 site file (`site_01.json`, `site_04.json`, `site_07.json`, or `site_11.json`). The loader checks frame, observer units/shape, and horizon coverage/valid flags before calculating vectors.


In [ ]:
SITE_JSON_PATH = Path('horizons/site_01.json')  # select site_01, site_04, site_07, or site_11
EXPECTED_FRAME = 'MOON_ME_DE421'
EXPECTED_SITE_LATLON = {
    'site01': (-89.45270, -137.1523),
    'site04': (-89.76205, -157.0750),
    'site07': (-88.80737, 123.7767),
    'site11': (-88.66832, -68.3754),
}
if not SITE_JSON_PATH.exists():
    raise FileNotFoundError(f'Upload Lucas schema v2 horizon JSON and set SITE_JSON_PATH: {SITE_JSON_PATH}')
site_doc = json.loads(SITE_JSON_PATH.read_text(encoding='utf-8'))
site_id = site_doc.get('site_id')
site_frame = site_doc.get('frame')
if site_doc.get('schema_version') != 2:
    raise ValueError(f"Expected horizon schema v2, got {site_doc.get('schema_version')!r}")
if site_frame != EXPECTED_FRAME:
    raise ValueError(f'Lucas JSON frame is {site_frame!r}; required frame is {EXPECTED_FRAME!r}. Do not fix by relabeling metadata.')
if site_doc.get('coverage_status') != 'complete':
    raise ValueError(f"Horizon coverage is {site_doc.get('coverage_status')!r}; stop if coverage is incomplete.")
valid = np.asarray(site_doc.get('valid', []), dtype=bool)
horizon = np.asarray(site_doc.get('horizon_deg', []), dtype=object)
if valid.shape != (7200,) or not valid.all() or horizon.shape != (7200,) or any(v is None for v in horizon):
    raise ValueError('Expected 7200 valid, non-null horizon samples at 0.05-degree spacing.')
if not np.isclose(float(site_doc.get('az_step_deg', np.nan)), 0.05) or int(site_doc.get('n_az', -1)) != 7200:
    raise ValueError('Unexpected azimuth grid; expected 7200 bins at 0.05 degrees.')
if not np.isclose(float(site_doc.get('observer_height_m', np.nan)), 1.0) or site_doc.get('observer_height_direction') != 'radial':
    raise ValueError('Expected Lucas 1 m radial observer position.')
observer_m = np.asarray(site_doc.get('observer_xyz_m'), dtype=float)
if observer_m.shape != (3,) or not np.isfinite(observer_m).all():
    raise ValueError('observer_xyz_m must contain three finite coordinates in metres.')
if not site_id:
    raise ValueError('Site ID is required in the JSON metadata.')
observer_km = observer_m / 1000.0  # SPKCPO observer position is in km
site_lat = float(site_doc['lat_deg'])
site_lon = float(site_doc['lon_deg'])
if str(site_id) in EXPECTED_SITE_LATLON:
    expected_lat, expected_lon = EXPECTED_SITE_LATLON[str(site_id)]
    if not (np.isclose(site_lat, expected_lat, atol=1e-5) and np.isclose(site_lon, expected_lon, atol=1e-4)):
        raise ValueError(f'{site_id} coordinates do not match Lucas updated ROI centroid; check that the newest JSON was used.')
print('Site:', site_id, '| frame:', site_frame, '| observer XYZ m:', observer_m, '| horizon valid:', int(valid.sum()), '/ 7200')


In [ ]:
UTC_START = '2026-11-14 00:00:00 UTC'
UTC_END = '2026-12-14 00:00:00 UTC'
STEP_S = 300
et_start = spice.str2et(UTC_START)
et_end = spice.str2et(UTC_END)
duration_s = et_end - et_start
n_intervals = round(duration_s / STEP_S)
if not np.isclose(n_intervals * STEP_S, duration_s, atol=1e-6):
    raise ValueError('Requested endpoints do not align with the 300-second cadence.')
# Spread samples uniformly in ET and include both UTC endpoints exactly; ET elapsed duration can differ from the nominal UTC day count by milliseconds.
times_s = np.linspace(0.0, duration_s, n_intervals + 1)
ets = et_start + times_s
assert len(times_s) == 8641 and np.isclose(times_s[-1], duration_s, atol=1e-9)
print('Samples:', len(ets), '| UTC:', UTC_START, 'to', UTC_END)

## Calculate observer-relative vectors
`spkcpo` uses the actual surface observer and returns the position relative to it. Do not subtract `observer_km` again. `NONE` may be run separately as a geometric comparison, but the exported vectors use `CN+S`.

In [ ]:
FRAME = 'MOON_ME_DE421'
ABCORR = 'CN+S'
CENTER = 'MOON'
OBSERVER_LOCATION = 'OBSERVER'

def body_vector_from_spice(target, et):
    state, _lt = spice.spkcpo(
        target, float(et), FRAME, OBSERVER_LOCATION, ABCORR,
        observer_km, CENTER, FRAME
    )
    return np.asarray(state[:3], dtype=float) * 1000.0  # km to m; preserve full length

def vectors_at(et):
    return {
        'sun': body_vector_from_spice('SUN', et),
        'earth': body_vector_from_spice('EARTH', et),
    }

sun_vectors_m = np.empty((len(ets), 3), dtype=float)
earth_vectors_m = np.empty((len(ets), 3), dtype=float)
for i, et in enumerate(ets):
    pair = vectors_at(et)
    sun_vectors_m[i] = pair['sun']
    earth_vectors_m[i] = pair['earth']
assert sun_vectors_m.shape == earth_vectors_m.shape == (8641, 3)
assert np.isfinite(sun_vectors_m).all() and np.isfinite(earth_vectors_m).all()
print('Sun vector lengths (m):', np.linalg.norm(sun_vectors_m, axis=1).min(), 'to', np.linalg.norm(sun_vectors_m, axis=1).max())
print('Earth vector lengths (m):', np.linalg.norm(earth_vectors_m, axis=1).min(), 'to', np.linalg.norm(earth_vectors_m, axis=1).max())

In [ ]:
# Optional diagnostics: azimuth 0° north, clockwise/east positive; elevations in degrees.
# Uses the site's planetocentric latitude/longitude if present. Vectors remain the required input.
lat_deg = site_lat
lon_deg = site_lon
def az_el_diagnostics(vectors_m):
    if lat_deg is None or lon_deg is None:
        print('No recognized lat/lon fields; skipping az/el diagnostics.')
        return None
    lat, lon = np.deg2rad([float(lat_deg), float(lon_deg)])
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([-np.sin(lat)*np.cos(lon), -np.sin(lat)*np.sin(lon), np.cos(lat)])
    up = np.array([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)])
    e, n, u = vectors_m @ east, vectors_m @ north, vectors_m @ up
    return {'azimuth_deg': np.mod(np.rad2deg(np.arctan2(e, n)), 360.0),
            'elevation_deg': np.rad2deg(np.arctan2(u, np.hypot(e, n)))}
sun_az_el = az_el_diagnostics(sun_vectors_m)
earth_az_el = az_el_diagnostics(earth_vectors_m)

# Terrain-only center-versus-horizon diagnostics for comparison with Lucas's profile.
horizon_az = np.asarray(site_doc['az_deg'], dtype=float)
horizon_deg = np.asarray(site_doc['horizon_deg'], dtype=float)
def horizon_at(azimuth_deg):
    az_ext = np.append(horizon_az, 360.0)
    h_ext = np.append(horizon_deg, horizon_deg[0])
    return np.interp(np.mod(azimuth_deg, 360.0), az_ext, h_ext)
sun_horizon_deg = horizon_at(sun_az_el['azimuth_deg'])
earth_horizon_deg = horizon_at(earth_az_el['azimuth_deg'])
sun_center_above_terrain_horizon = sun_az_el['elevation_deg'] >= sun_horizon_deg
earth_center_above_terrain_horizon = earth_az_el['elevation_deg'] >= earth_horizon_deg


In [ ]:
KERNEL_FILENAMES = list(KERNEL_URLS)
metadata = {
    'site_id': str(site_id),
    'observer_position_xyz_m': observer_m.tolist(),
    'frame': FRAME, 'center': CENTER, 'abcorr': ABCORR,
    'spkcpo_refloc': OBSERVER_LOCATION,
    'kernel_filenames': KERNEL_FILENAMES,
    'start_utc': '2026-11-14T00:00:00Z',
    'end_utc': '2026-12-14T00:00:00Z',
    'time_scale': 'ET seconds elapsed from et_start',
    'cadence_seconds': STEP_S, 'sample_count': int(len(times_s)),
    'vector_units': 'm', 'vector_origin': 'surface observer',
    'vector_target': 'body center', 'vectors_normalized': False,
    'sun_key': 'sun_vectors_m', 'earth_key': 'earth_vectors_m',
}
out_stem = f'ephemeris_{site_id}'
np.savez_compressed(out_stem + '.npz', times_s=times_s,
                    sun_vectors_m=sun_vectors_m, earth_vectors_m=earth_vectors_m)
Path(out_stem + '.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

# Lucas requested a 1–2 day comparison. Export the first 48 hours of optional az/el diagnostics.
n_48h = int(48 * 3600 / STEP_S) + 1
if sun_az_el is not None and earth_az_el is not None:
    import csv
    diag_path = out_stem + '_az_el_first_48h.csv'
    with open(diag_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['utc', 'elapsed_et_s', 'sun_azimuth_deg', 'sun_elevation_deg', 'sun_terrain_horizon_deg', 'sun_center_above_terrain_horizon', 'earth_azimuth_deg', 'earth_elevation_deg', 'earth_terrain_horizon_deg', 'earth_center_above_terrain_horizon'])
        for i in range(min(n_48h, len(times_s))):
            utc = spice.et2utc(float(ets[i]), 'ISOC', 3) + 'Z'
            writer.writerow([utc, times_s[i], sun_az_el['azimuth_deg'][i], sun_az_el['elevation_deg'][i], sun_horizon_deg[i], bool(sun_center_above_terrain_horizon[i]), earth_az_el['azimuth_deg'][i], earth_az_el['elevation_deg'][i], earth_horizon_deg[i], bool(earth_center_above_terrain_horizon[i])])
    print('Wrote 48-hour diagnostic:', diag_path)
else:
    print('Az/el/horizon diagnostic CSV skipped; site coordinates are unavailable.')
print('Wrote', out_stem + '.npz', 'and', out_stem + '.json')

## Handoff and limitations
Send the `.npz` and metadata sidecar to Zarina. Visibility rules and final windows/metrics belong to her `analyze` function: Earth center above terrain horizon and Sun fraction at least 0.80, including terrain obstruction and Earth occultation. Use `vectors_at(et)` at bracketing/refinement steps to help locate transitions to approximately 30–60 seconds; final windows/metrics still come from `analyze`.

The notebook exports the first 48 hours of Sun/Earth azimuth and elevation plus the terrain horizon at each azimuth to a CSV for Lucas's comparison. Its center-versus-horizon flags are diagnostic only; the Sun-disc fraction rule and final windows/metrics belong to Zarina's analyze function, including terrain obstruction and Earth occultation. Zarina confirmed the keys `sun_vectors_m` and `earth_vectors_m`, full observer-relative vectors in metres, and `times_s` as elapsed ET seconds from the first sample; UTC bounds are recorded in the sidecar.